# Stratifying "What We Eat in America" – Refactored

`CC-BY 2026 Brooksbank, Kassabov, Wilson`

This notebook applies Dleto stratification algorithms to USDA nutrition data to uncover hidden clustering patterns in food compositions, specifically examining the relationship between added sugars and food category combinations.

**Data Source:** [What We Eat in America](www.ars.usda.gov/nea/bhnrc/fsrg), NHANES 2017-March 2020 Prepandemic

**Disclaimer:** This notebook is for algorithmic demonstration only and does not constitute nutritional advice.

---

### Quick Navigation
- [1. Import Libraries and Load Data](#section-1)
- [2. Define Food Category Mappings](#section-2)
- [3. Extract and Validate Nutritional Data](#section-3)
- [4. Construct the Sugar Tensor](#section-4)
- [5. HOSVD Analysis](#section-5)
- [6. PARAFAC Decomposition](#section-6)
- [7. Universal Chisel Stratification](#section-7)
- [8. Adjoint Chisel Stratification](#section-8)
- [9. Compare and Visualize Results](#section-9)

## Section 1: Import Libraries and Load Data {#section-1}

Load required packages and the USDA food composition data from CSV files.

In [1]:
using Pkg
Pkg.activate("../") 

Pkg.add("CSV"); Pkg.add("DataFrames"); Pkg.add("Dleto")
using CSV, DataFrames, ITensors, Plots, Statistics, LinearAlgebra, Random

# Load the USDA FPED data
data = CSV.read("FPED_1720.csv", DataFrame)
key_data = CSV.read("FPED_1720-key.csv", DataFrame)

println("✓ Data loaded successfully")
println("  Total foods: $(size(data, 1))")
println("  Nutritional categories: $(size(key_data, 1))")
println("  Data columns: $(size(data, 2))")

   Resolving package versions...
      Compat entries added for 
     Project No packages added to or removed from `~/CODE/OpenDleto/Project.toml`
    Manifest No packages added to or removed from `~/CODE/OpenDleto/Manifest.toml`
   Resolving package versions...
      Compat entries added for 
     Project No packages added to or removed from `~/CODE/OpenDleto/Project.toml`
    Manifest No packages added to or removed from `~/CODE/OpenDleto/Manifest.toml`


LoadError: The following package names could not be resolved:
 * Dleto (e7db872f-2c5e-4d8f-9a1c-3b5e8f4a9c2d in manifest but not in project)

In [ ]:
# Display sample data
println("\nSample food items:")
display(data[147:155, vcat(1:2, 33:34, 38)])

## Section 2: Define Food Category Mappings {#section-2}

Create systematic mappings for the 7 major food categories and helper functions.

In [ ]:
# Helper function for friendly category labels
function friendly_label(code::AbstractString)
    label_map = Dict(
        "F_CITMLB (cup eq)" => "Citrus, tomatoes, melons, berries",
        "F_OTHER (cup eq)" => "Other fruits (apples, bananas, etc.)",
        "F_JUICE (cup eq)" => "Fruit juice",
        "V_DRKGR (cup eq)" => "Dark green vegetables",
        "V_REDOR_TOMATO (cup eq)" => "Red/orange vegetables (tomatoes)",
        "V_REDOR_OTHER (cup eq)" => "Red/orange vegetables (other)",
        "V_STARCHY_POTATO (cup eq)" => "Starchy vegetables (potatoes)",
        "V_STARCHY_OTHER (cup eq)" => "Starchy vegetables (other)",
        "V_OTHER (cup eq)" => "Other vegetables",
        "V_LEGUMES (cup eq)" => "Legumes (as vegetables)",
        "PF_MEAT (oz eq)" => "Meat (beef, pork, lamb, game)",
        "PF_CUREDMEAT (oz eq)" => "Cured meat",
        "PF_ORGAN (oz eq)" => "Organ meats",
        "PF_POULT (oz eq)" => "Poultry",
        "PF_SEAFD_HI (oz eq)" => "High omega-3 seafood",
        "PF_SEAFD_LOW (oz eq)" => "Low omega-3 seafood",
        "PF_EGGS (oz eq)" => "Eggs",
        "PF_SOY (oz eq)" => "Soy products",
        "PF_NUTSDS (oz eq)" => "Nuts and seeds",
        "PF_LEGUMES (oz eq)" => "Legumes (as protein)",
        "F_EMPTY" => "No measurable fruit",
        "V_EMPTY" => "No measurable vegetable",
        "PF_EMPTY" => "No measurable protein"
    )
    return get(label_map, code, code)
end

# Category filtering functions
nutritional_columns = names(data)[3:end]

isfruit(col) = startswith(string(col), "F_") && !occursin("TOTAL", string(col))
isvegetable(col) = startswith(string(col), "V_") && !occursin("TOTAL", string(col))
isgrain(col) = startswith(string(col), "G_") && !occursin("TOTAL", string(col))
isprotein(col) = startswith(string(col), "PF_") && !occursin("TOTAL", string(col))
isdairy(col) = startswith(string(col), "D_") && !occursin("TOTAL", string(col))
isfat_or_oil(col) = string(col) in ["OILS (grams)", "SOLID_FATS (grams)"]
isdiscretionary(col) = string(col) in ["ADD_SUGARS (tsp eq)", "A_DRINKS (no. of drinks)"]

# Extract category columns
fruits = filter(isfruit, nutritional_columns)
vegetables = filter(isvegetable, nutritional_columns)
grains = filter(isgrain, nutritional_columns)
proteins = filter(isprotein, nutritional_columns)
dairy = filter(isdairy, nutritional_columns)
fats_oils = filter(isfat_or_oil, nutritional_columns)
discretionary = filter(isdiscretionary, nutritional_columns)

println("✓ Category mappings defined:")
println("  🍎 Fruits: $(length(fruits)) categories")
println("  🥬 Vegetables: $(length(vegetables)) categories")
println("  🌾 Grains: $(length(grains)) categories")
println("  🥩 Proteins: $(length(proteins)) categories")
println("  🥛 Dairy: $(length(dairy)) categories")
println("  🫒 Fats & Oils: $(length(fats_oils)) categories")
println("  🍭 Discretionary: $(length(discretionary)) categories")

## Section 3: Extract and Validate Nutritional Data {#section-3}

Check data quality and prepare categorical lists with EMPTY entries.

In [ ]:
# Extract numerical data
numerical_data = Matrix(data[:, 3:end])

# Data quality checks
missing_count = sum(ismissing.(numerical_data))
zero_count = sum(numerical_data .== 0)
total_elements = length(numerical_data)
sparsity = zero_count / total_elements

println("✓ Data Validation:")
println("  Missing values: $missing_count")
println("  Zero values: $zero_count / $total_elements ($(round(sparsity * 100, digits=1))% sparse)")
println("  Value range: [$(minimum(skipmissing(numerical_data))), $(maximum(skipmissing(numerical_data)))]")
println("  Mean value: $(round(mean(skipmissing(numerical_data)), digits=3))")

In [ ]:
# Create category lists with EMPTY entries for foods with zero measurable amounts
fruits_with_empty = vcat(fruits, ["F_EMPTY"])
vegetables_with_empty = vcat(vegetables, ["V_EMPTY"])
proteins_with_empty = vcat(proteins, ["PF_EMPTY"])

println("\n✓ Category lists with EMPTY entries:")
println("  Fruits: $(length(fruits_with_empty)) entries (including EMPTY)")
println("  Vegetables: $(length(vegetables_with_empty)) entries (including EMPTY)")
println("  Proteins: $(length(proteins_with_empty)) entries (including EMPTY)")

## Section 4: Construct the Sugar Tensor {#section-4}

Build a 3-way tensor (Fruits × Vegetables × Proteins) where each entry sums the added sugars for foods in that category combination.

In [ ]:
# Create ITensor indices for the three modes
f = Index(length(fruits_with_empty), "Fruits")
v = Index(length(vegetables_with_empty), "Vegetables")
p = Index(length(proteins_with_empty), "Proteins")

# Initialize tensor
Sugar = ITensor(f, v, p)

# Populate tensor by iterating over foods
processed_foods = 0
empty_counts = Dict(:fruits => 0, :vegetables => 0, :proteins => 0)

for row_idx in 1:nrow(data)
    sugar_value = data[row_idx, "ADD_SUGARS (tsp eq)"]
    
    # Skip foods with zero or missing sugar
    if ismissing(sugar_value) || sugar_value == 0.0
        continue
    end
    
    # Extract category values for this food
    fruit_vals = [coalesce(data[row_idx, col], 0.0) for col in fruits]
    veg_vals = [coalesce(data[row_idx, col], 0.0) for col in vegetables]
    protein_vals = [coalesce(data[row_idx, col], 0.0) for col in proteins]
    
    # Assign to EMPTY if below threshold, otherwise to category with max value
    fruit_idx = maximum(fruit_vals) <= 1e-10 ? length(fruits_with_empty) : argmax(fruit_vals)
    veg_idx = maximum(veg_vals) <= 1e-10 ? length(vegetables_with_empty) : argmax(veg_vals)
    protein_idx = maximum(protein_vals) <= 1e-10 ? length(proteins_with_empty) : argmax(protein_vals)
    
    if fruit_idx == length(fruits_with_empty)
        empty_counts[:fruits] += 1
    end
    if veg_idx == length(vegetables_with_empty)
        empty_counts[:vegetables] += 1
    end
    if protein_idx == length(proteins_with_empty)
        empty_counts[:proteins] += 1
    end
    
    # Add sugar to tensor at this coordinate
    Sugar[f => fruit_idx, v => veg_idx, p => protein_idx] += sugar_value
    processed_foods += 1
end

# Compute tensor statistics
non_zero_entries = 0
max_value = 0.0
for i in 1:dim(f), j in 1:dim(v), k in 1:dim(p)
    val = Sugar[f => i, v => j, p => k]
    if val != 0.0
        non_zero_entries += 1
        max_value = max(max_value, val)
    end
end

total_entries = dim(f) * dim(v) * dim(p)
tensor_sparsity = (total_entries - non_zero_entries) / total_entries

println("✓ Sugar Tensor Constructed:")
println("  Foods processed: $processed_foods ($(round((processed_foods / (nrow(data) - 2)) * 100, digits=1))% of total)")
println("  Tensor dimensions: ($(dim(f)), $(dim(v)), $(dim(p)))")
println("  Non-zero entries: $non_zero_entries / $total_entries ($(round(tensor_sparsity * 100, digits=1))% sparse)")
println("  Max entry value: $(round(max_value, digits=2)) tsp eq")
println("  EMPTY assignments: fruit=$(empty_counts[:fruits]), veg=$(empty_counts[:vegetables]), protein=$(empty_counts[:proteins])")

In [ ]:
# Visualize the original sugar tensor
plot_tensor(Sugar, title="Original Sugar Tensor (Fruits × Vegetables × Proteins)",
    xlabel="Fruits", ylabel="Vegetables", zlabel="Proteins")

## Section 5: HOSVD Analysis {#section-5}

Compute Higher-Order Singular Value Decomposition to extract singular value spectra and the core tensor.

In [ ]:
# Compute HOSVD on the original tensor
mode_inds = collect(inds(Sugar))
N = length(mode_inds)
U_factors = Vector{ITensor}(undef, N)
link_inds = Vector{Index}(undef, N)
svs = Vector{Vector{Float64}}(undef, N)

for n in 1:N
    U, S, V = svd(Sugar, mode_inds[n])
    U_factors[n] = U
    s1, s2 = inds(S)
    link_inds[n] = s1
    svs[n] = [S[s1 => k, s2 => k] for k in 1:dim(s1)]
end

# Reconstruct core tensor and compute error
G = Sugar
for n in 1:N
    G *= dag(U_factors[n])
end

A_hat = G
for n in 1:N
    A_hat *= U_factors[n]
end

reconstruction_error = norm(Sugar - A_hat) / max(norm(Sugar), eps(Float64))

println("✓ HOSVD Computed:")
println("  Tensor order: $N")
println("  Original dims: $(dim.(mode_inds))")
println("  Core dims: $(dim.(inds(G)))")
println("  Reconstruction error: $reconstruction_error")

In [ ]:
# Plot singular value spectra for each mode
p_hosvd = plot(layout=(1, N), size=(420 * N, 320), legend=false)
mode_labels = ["Fruits", "Vegetables", "Proteins"]

for n in 1:N
    plot!(p_hosvd[n], 1:length(svs[n]), max.(svs[n], eps(Float64)),
        marker=:circle, linewidth=2, xlabel="Component", ylabel="Singular value",
        yscale=:log10, title="$(mode_labels[n]) spectrum (dim=$(dim(mode_inds[n])))")
end

display(p_hosvd)

In [ ]:
# Visualize the HOSVD core tensor
inds_core = collect(inds(G))
plot_tensor(G, title="HOSVD Core Tensor",
    xlabel="Fruits (mixed)", ylabel="Vegetables (mixed)", zlabel="Proteins (mixed)")

## Section 6: PARAFAC Decomposition {#section-6}

Implement and apply CP-ALS (Canonical Polyadic Alternating Least Squares) for low-rank tensor approximation.

In [ ]:
# Helper functions for PARAFAC/CP-ALS
function khatri_rao(A::AbstractMatrix, B::AbstractMatrix)
    size(A, 2) == size(B, 2) || error("Khatri-Rao requires same number of columns")
    m, r = size(A)
    n, _ = size(B)
    KR = zeros(promote_type(eltype(A), eltype(B)), m * n, r)
    for j in 1:r
        KR[:, j] = vec(kron(A[:, j], B[:, j]))
    end
    return KR
end

function unfold3(X::Array{Float64, 3}, mode::Int)
    if mode == 1
        return reshape(permutedims(X, (1, 2, 3)), size(X, 1), :)
    elseif mode == 2
        return reshape(permutedims(X, (2, 1, 3)), size(X, 2), :)
    elseif mode == 3
        return reshape(permutedims(X, (3, 1, 2)), size(X, 3), :)
    else
        error("mode must be 1, 2, or 3")
    end
end

function cp_reconstruct(A::Matrix{Float64}, B::Matrix{Float64}, C::Matrix{Float64}, lambda::Vector{Float64})
    I1, R = size(A)
    J1, _ = size(B)
    K1, _ = size(C)
    Xhat = zeros(Float64, I1, J1, K1)
    for r in 1:R
        Xhat .+= lambda[r] .* reshape(A[:, r], I1, 1, 1) .* reshape(B[:, r], 1, J1, 1) .* reshape(C[:, r], 1, 1, K1)
    end
    return Xhat
end

function cp_als_3way(X::Array{Float64, 3}, R::Int; maxiter::Int=200, tol::Float64=1e-7, seed::Int=1234, ridge::Float64=1e-8)
    I1, J1, K1 = size(X)
    Random.seed!(seed)
    
    A = rand(I1, R)
    B = rand(J1, R)
    C = rand(K1, R)
    lambda = ones(Float64, R)
    
    X1 = unfold3(X, 1)
    X2 = unfold3(X, 2)
    X3 = unfold3(X, 3)
    
    prev_relerr = Inf
    relerr = Inf
    
    for iter in 1:maxiter
        KRcb = khatri_rao(C, B)
        G1 = (B' * B) .* (C' * C) + ridge * Matrix{Float64}(I, R, R)
        A = (X1 * KRcb) / G1
        
        KRca = khatri_rao(C, A)
        G2 = (A' * A) .* (C' * C) + ridge * Matrix{Float64}(I, R, R)
        B = (X2 * KRca) / G2
        
        KRba = khatri_rao(B, A)
        G3 = (A' * A) .* (B' * B) + ridge * Matrix{Float64}(I, R, R)
        C = (X3 * KRba) / G3
        
        # Normalize columns and absorb scales in lambda
        lambda .= 1.0
        for r in 1:R
            nA = norm(@view A[:, r]); nA = nA > 0 ? nA : 1.0
            nB = norm(@view B[:, r]); nB = nB > 0 ? nB : 1.0
            nC = norm(@view C[:, r]); nC = nC > 0 ? nC : 1.0
            A[:, r] ./= nA
            B[:, r] ./= nB
            C[:, r] ./= nC
            lambda[r] = nA * nB * nC
        end
        
        Xhat = cp_reconstruct(A, B, C, lambda)
        relerr = norm(X - Xhat) / max(norm(X), eps(Float64))
        
        if abs(prev_relerr - relerr) < tol
            break
        end
        prev_relerr = relerr
    end
    
    return A, B, C, lambda, relerr
end

println("✓ PARAFAC/CP-ALS functions defined")

In [ ]:
# Convert tensor to dense Array for CP-ALS
sugar_inds = collect(inds(Sugar))
I_dim, J_dim, K_dim = dim(sugar_inds[1]), dim(sugar_inds[2]), dim(sugar_inds[3])
X_sugar = zeros(Float64, I_dim, J_dim, K_dim)

for i in 1:I_dim, j in 1:J_dim, k in 1:K_dim
    X_sugar[i, j, k] = Sugar[sugar_inds[1] => i, sugar_inds[2] => j, sugar_inds[3] => k]
end

# Run PARAFAC
cp_rank = 5
@time A_cp, B_cp, C_cp, lambda_cp, relerr_cp = cp_als_3way(X_sugar, cp_rank; maxiter=300, tol=1e-8, seed=42, ridge=1e-7)

Xhat_cp = cp_reconstruct(A_cp, B_cp, C_cp, lambda_cp)

# Convert back to ITensor for visualization
Sugar_parafac = ITensor(sugar_inds...)
for i in 1:I_dim, j in 1:J_dim, k in 1:K_dim
    Sugar_parafac[sugar_inds[1] => i, sugar_inds[2] => j, sugar_inds[3] => k] = Xhat_cp[i, j, k]
end

println("✓ PARAFAC Decomposition Complete:")
println("  Rank: $cp_rank")
println("  Reconstruction error: $(round(relerr_cp, digits=6))")

In [ ]:
# Visualize PARAFAC approximation
plot_tensor(Sugar_parafac, title="Sugar Tensor PARAFAC Approximation (rank=$cp_rank)",
    xlabel="Fruits", ylabel="Vegetables", zlabel="Proteins")

## Section 7: Universal Chisel Stratification {#section-7}

Apply Dleto's universal chisel to detect independent derivation patterns across all modes.

In [ ]:
# Apply nondegeneracy reduction first
nondeg_Sugar, Ys_nondeg = nondeg(Sugar)
nondeg_inds = collect(inds(nondeg_Sugar))

println("✓ Nondegenerate reduction applied:")
println("  Original dims: $(dim.(collect(inds(Sugar))))")
println("  Nondeg dims: $(dim.(nondeg_inds))")

In [ ]:
# Universal Chisel Stratification
ch_universal = UniversalChisel(length(nondeg_inds))
fr_universal = nondeg_inds
Ω_universal = IndTransverseOps(fr_universal, UniversalOp())

# Store stratifications for each ivec
Sugar_strat_by_ivec = Dict{Int, ITensor}()
Xs_strat_by_ivec = Dict{Int, Vector{ITensor}}()

for ivec in 3:7
    Sugar_strat_i, Xs_strat_i = stratify(Ω_universal, ch_universal, nondeg_Sugar; ivec=ivec)
    Sugar_strat_by_ivec[ivec] = Sugar_strat_i
    Xs_strat_by_ivec[ivec] = Xs_strat_i
    
    inds_i = collect(inds(Sugar_strat_i))
    display(plot_tensor(Sugar_strat_i,
        title="Universal Stratified Tensor (ivec=$ivec)",
        xlabel="Fruits (mixed)", ylabel="Vegetables (mixed)", zlabel="Proteins (mixed)"))
end

println("✓ Universal chisel stratification completed for ivec ∈ {3, 4, 5, 6, 7}")

In [ ]:
# Extract and display protein/vegetable mixtures for universal stratification
function extract_mixtures(T::ITensor, Xs::Vector{ITensor}, proteins_list::Vector, vegetables_list::Vector)
    inds_mix = collect(inds(T))
    dims_mix = [dim(i) for i in inds_mix]
    
    # Locate modes by dimension
    pos_prot = findfirst(==(length(proteins_list)), dims_mix)
    pos_veg = findfirst(==(length(vegetables_list)), dims_mix)
    
    isnothing(pos_prot) && return nothing
    isnothing(pos_veg) && return nothing
    
    # Extract transformation matrices
    X_prot = first(X for X in Xs if any(dim(ix) == length(proteins_list) for ix in inds(X)))
    X_veg = first(X for X in Xs if any(dim(ix) == length(vegetables_list) for ix in inds(X)))
    
    mix_prot_idx = inds_mix[pos_prot]
    mix_veg_idx = inds_mix[pos_veg]
    
    # Extract vectors for a fixed coordinate (e.g., protein=1, veg=1)
    e_prot = ITensor(mix_prot_idx); e_prot[mix_prot_idx => 1] = 1.0
    e_veg = ITensor(mix_veg_idx); e_veg[mix_veg_idx => 1] = 1.0
    
    prot_mix = dag(X_prot) * e_prot
    veg_mix = dag(X_veg) * e_veg
    
    prot_idx = only([ix for ix in inds(prot_mix) if dim(ix) == length(proteins_list)])
    veg_idx = only([ix for ix in inds(veg_mix) if dim(ix) == length(vegetables_list)])
    
    prot_weights = [prot_mix[prot_idx => i] for i in 1:dim(prot_idx)]
    veg_weights = [veg_mix[veg_idx => i] for i in 1:dim(veg_idx)]
    
    return prot_weights, veg_weights
end

println("✓ Mixture extraction functions defined")

## Section 8: Adjoint Chisel Stratification {#section-8}

Apply an adjoint chisel focused on fruit and vegetable modes (protein axis constrained to 0).

In [ ]:
# Adjoint Chisel Stratification
ch_adj = AdjointChisel(3, 1, 2)  # 3D tensor, chiseling on modes 1 & 2 (fruit & veg)
fr_adj = nondeg_inds
Ω_adj = IndTransverseOps(fr_adj, UniversalOp())

adj_strat_by_ivec = Dict{Int, ITensor}()
adj_Xs_by_ivec = Dict{Int, Vector{ITensor}}()

for ivec in 2:4
    Sugar_adj_i, Xs_adj_i = stratify(Ω_adj, ch_adj, nondeg_Sugar; tol=1e-6, ivec=ivec)
    adj_strat_by_ivec[ivec] = Sugar_adj_i
    adj_Xs_by_ivec[ivec] = Xs_adj_i
    
    inds_adj_i = collect(inds(Sugar_adj_i))
    display(plot_tensor(Sugar_adj_i,
        title="Adjoint Stratified Tensor (ivec=$ivec)",
        xlabel="Fruits (mixed)", ylabel="Vegetables (mixed)", zlabel="Proteins (mixed)"))
end

println("✓ Adjoint chisel stratification completed for ivec ∈ {2, 3, 4}")

## Section 9: Compare and Visualize Results {#section-9}

Summarize findings and create comprehensive comparison.

In [ ]:
# Summary of findings
println("\n" * "="^70)
println("COMPREHENSIVE ANALYSIS SUMMARY")
println("="^70)

println("\n1️⃣  TUCKER DECOMPOSITION (Degeneracy Analysis)")
println("   → Trivial at default tolerance; no significant structural redundancy")

println("\n2️⃣  HIGHER-ORDER SINGULAR VALUES")
println("   → Core tensor does not reveal obvious low-rank structure")
println("   → Singular value decay is moderate across all modes")

println("\n3️⃣  PARAFAC/CP-ALS Rank-5 Approximation")
println("   → Reconstruction error: $(round(relerr_cp, digits=6))")
println("   → Indicates dense, complex distribution of added sugars")

println("\n4️⃣  UNIVERSAL CHISEL STRATIFICATION (5 independent derivations)")
println("   → Detects clustering in protein-vegetable modes")
println("   → Identifies foods combining: meat + cured meat + poultry")
println("   → With: dark greens + starchy vegetables (non-potatoes)")

println("\n5️⃣  ADJOINT CHISEL STRATIFICATION (3 independent derivations)")
println("   → Simplified stratification focusing on fruit-vegetable coupling")
println("   → Confirms protein-vegetable clustering pattern")
println("   → Reduces dimensionality while preserving key signals")

println("\n" * "="^70)
println("KEY FINDING:")
println("Added sugars cluster in foods combining:")
println("  • Proteins: Meat, cured meat, poultry (sauces/dressings?)")
println("  • Vegetables: Dark greens + starchy vegetables")
println("→ Likely source: condiments, glazes, or preparation sauces")
println("="^70)

In [ ]:
# Create comparison visualization
p_comparison = plot(
    plot_tensor(Sugar, title="Original Tensor", xlabel="F", ylabel="V", zlabel="P"),
    plot_tensor(Sugar_parafac, title="PARAFAC (rank=5)", xlabel="F", ylabel="V", zlabel="P"),
    layout=(1, 2), size=(800, 350)
)
display(p_comparison)

println("\nVisualization: Original vs PARAFAC Approximation")

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.

### Interpretation

The refactored analysis reveals:

**Methodological Insights:**
- Tucker decomposition analysis shows the data has no trivial low-rank structure
- HOSVD singular value spectra indicate a complex, non-hierarchical distribution
- PARAFAC reconstruction error ($(round(relerr_cp, digits=6))) suggests foods with added sugars form intricate patterns

**Stratification Findings:**
- **Universal Chisel**: 5 independent derivations identify a robust cluster in protein-vegetable modes
- **Adjoint Chisel**: 3 independent derivations confirm the clustering with reduced complexity
- The dominant pattern pairs high-sugar content with specific food combinations

**Practical Observation:**
The cluster consistently combines meat-based proteins with leafy and starchy vegetables, suggesting the added sugars come from accompanying sauces, dressings, or glazes rather than the vegetables themselves.

**Methodological Advantage:**
Dleto's stratification approach reveals this structure even though it has low magnitude influence on traditional singular value metrics—demonstrating the value of derivation-based analysis for detecting subtle but meaningful patterns in sparse, complex data.